<a href="https://colab.research.google.com/github/e23378-Tharz/Statistical-Learning-e23378/blob/main/Kalman_filter_assignment_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Kalman Filter Assignment — Full Solution

---
# Q1 — Analytical Derivation

Consider the filter model
$$x^{-}_k = A_{k-1}\,x^{+}_{k-1} + G_{k-1}\,w_{k-1}, \qquad y^{-}_k = H_k\,x^{-}_k + z_k$$
where $x^{+}_{k-1} \sim \mathcal{N}(m_{k-1}, P_{k-1})$, $w_{k-1} \sim \mathcal{N}(0,\Sigma_p)$, $z_k \sim \mathcal{N}(0,\Sigma_m)$, all independent.

### Part 1 — Predicted state distribution

**Claim:** $x_k^- \sim \mathcal{N}(m_k^-, P_k^-)$ where $m_k^- = A_{k-1}m_{k-1}$ and $P_k^- = A_{k-1}P_{k-1}A_{k-1}^\top + G_{k-1}\Sigma_p G_{k-1}^\top$.

**Proof:**

$x_k^-$ is an affine function of independent Gaussians $x^+_{k-1}$ and $w_{k-1}$, hence Gaussian.

**Mean:**
$$m_k^- = \mathbb{E}[x_k^-] = A_{k-1}\,\mathbb{E}[x^+_{k-1}] + G_{k-1}\,\mathbb{E}[w_{k-1}] = A_{k-1}m_{k-1} + 0 = A_{k-1}m_{k-1}$$

**Covariance** (using independence of $x^+_{k-1}$ and $w_{k-1}$, and $\mathbb{E}[w_{k-1}]=0$):
$$P_k^- = \text{Var}(A_{k-1}x^+_{k-1} + G_{k-1}w_{k-1})$$
$$= A_{k-1}\,\text{Var}(x^+_{k-1})\,A_{k-1}^\top + G_{k-1}\,\text{Var}(w_{k-1})\,G_{k-1}^\top$$
$$= A_{k-1}P_{k-1}A_{k-1}^\top + G_{k-1}\Sigma_p G_{k-1}^\top \qquad \blacksquare$$

### Part 2 — Predicted measurement distribution

**Claim:** $y_k^- \sim \mathcal{N}(H_k m_k^-,\; H_k P_k^- H_k^\top + \Sigma_m)$.

**Proof:**

$y_k^- = H_k x_k^- + z_k$ is again an affine function of independent Gaussians $x_k^-$ and $z_k$, hence Gaussian.

**Mean:**
$$\mathbb{E}[y_k^-] = H_k\,\mathbb{E}[x_k^-] + \mathbb{E}[z_k] = H_k m_k^-$$

**Variance** (independence of $x_k^-$ and $z_k$):
$$\text{Var}(y_k^-) = H_k\,\text{Var}(x_k^-)\,H_k^\top + \text{Var}(z_k) = H_k P_k^- H_k^\top + \Sigma_m \qquad \blacksquare$$

### Part 3 — Joint distribution

**Claim:**
$$\begin{bmatrix} x_k^- \\ y^{-}_k \end{bmatrix} \sim \mathcal{N}\!\left(
\begin{bmatrix} m_k^- \\ H_k m_k^- \end{bmatrix},
\begin{bmatrix} P_k^- & P_k^- H_k^\top \\ H_k P_k^- & H_k P_k^- H_k^\top + \Sigma_m \end{bmatrix}
\right)$$

**Proof:**

Parts 1 and 2 give the diagonal blocks. It remains to compute the cross-covariance:
$$\text{Cov}(x_k^-, y_k^-) = \mathbb{E}[(x_k^- - m_k^-)(y_k^- - H_k m_k^-)^\top]$$
$$= \mathbb{E}[(x_k^- - m_k^-)(H_k(x_k^- - m_k^-) + z_k)^\top]$$
$$= \mathbb{E}[(x_k^- - m_k^-)(x_k^- - m_k^-)^\top]\,H_k^\top + \mathbb{E}[(x_k^- - m_k^-)z_k^\top]$$
$$= P_k^- H_k^\top + 0 = P_k^- H_k^\top$$

where the cross-term vanishes because $x_k^-$ and $z_k$ are independent. The joint vector is an affine transform of independent Gaussians, hence jointly Gaussian. $\blacksquare$

### Part 4 — Measurement update (posterior)

**Claim:** $(x_k^- \mid y_k^- = y_k^\text{obs}) \sim \mathcal{N}(m_k, P_k)$ with:
$$K_k = P_k^- H_k^\top (H_k P_k^- H_k^\top + \Sigma_m)^{-1}, \quad m_k = m_k^- + K_k(y_k^\text{obs} - H_k m_k^-), \quad P_k = (I-K_kH_k)P_k^-$$

**Proof:**

Use the standard **Gaussian conditioning formula**: for a joint Gaussian
$$\begin{bmatrix}u\\v\end{bmatrix} \sim \mathcal{N}\!\left(\begin{bmatrix}\mu_u\\\mu_v\end{bmatrix}, \begin{bmatrix}\Sigma_{uu}&\Sigma_{uv}\\\Sigma_{vu}&\Sigma_{vv}\end{bmatrix}\right)$$
the conditional is
$$(u \mid v=v_0) \sim \mathcal{N}\!\left(\mu_u + \Sigma_{uv}\Sigma_{vv}^{-1}(v_0-\mu_v),\;\Sigma_{uu}-\Sigma_{uv}\Sigma_{vv}^{-1}\Sigma_{vu}\right)$$

Identifying $u=x_k^-$, $v=y_k^-$, $\mu_u=m_k^-$, $\mu_v=H_km_k^-$, $\Sigma_{uu}=P_k^-$, $\Sigma_{uv}=P_k^-H_k^\top$, $\Sigma_{vv}=H_kP_k^-H_k^\top+\Sigma_m$:

$$K_k = \Sigma_{uv}\Sigma_{vv}^{-1} = P_k^- H_k^\top(H_kP_k^-H_k^\top+\Sigma_m)^{-1}$$
$$m_k = m_k^- + K_k(y_k^\text{obs} - H_km_k^-)$$
$$P_k = P_k^- - P_k^-H_k^\top(H_kP_k^-H_k^\top+\Sigma_m)^{-1}H_kP_k^- = (I-K_kH_k)P_k^- \qquad \blacksquare$$

### Part 5 — Conditional mean and variance

Directly from Part 4:

$$\mathbb{E}[x_k^- \mid y_k^- = y_k^\text{obs}] = m_k = m_k^- + K_k(y_k^\text{obs} - H_km_k^-)$$

$$\text{Var}(x_k^- \mid y_k^- = y_k^\text{obs}) = P_k = (I - K_kH_k)P_k^-$$

**Interpretation:** $m_k$ is the MMSE (minimum mean-square error) estimate of $x_k$ given all measurements up to time $k$. $P_k$ quantifies remaining uncertainty after assimilating $y_k^\text{obs}$. The update shrinks uncertainty: $P_k \preceq P_k^-$ (in the positive semidefinite sense).

---
# Q2 — 1-D Scalar Example

Filter model: $x_k^- = a\,x_{k-1}^+ + w_{k-1}$, $y_k^- = h\,x_k^- + z_k$, with $w \sim \mathcal{N}(0,q)$, $z \sim \mathcal{N}(0,r)$.

### Part 1 — Scalar prediction recurrence

**Claim:** $m_k^- = a\,m_{k-1}$, $P_k^- = a^2 P_{k-1} + q$.

**Proof:** Set $A=a$ (scalar), $G=1$, $\Sigma_p=q$ in the general formula:
$$m_k^- = a\,m_{k-1}, \qquad P_k^- = a^2 P_{k-1} + 1^2 \cdot q = a^2P_{k-1}+q \qquad\blacksquare$$

### Part 2 — Scalar update

**Claim:**
$m_k = m_k^- + \frac{P_k^- h}{S_k}(y_k^\text{obs} - h\,m_k^-)$, $P_k = \left(1-\frac{P_k^- h^2}{S_k}\right)P_k^-$, where $S_k = h^2 P_k^- + r$.

**Proof:** With scalars $H=h$, $\Sigma_m=r$:
$$K_k = \frac{P_k^- h}{h^2 P_k^- + r} = \frac{P_k^- h}{S_k}$$
$$m_k = m_k^- + K_k(y_k^\text{obs} - h\,m_k^-) = m_k^- + \frac{P_k^-h}{S_k}(y_k^\text{obs}-h\,m_k^-)$$
$$P_k = (1 - K_k h)P_k^- = \left(1 - \frac{P_k^-h^2}{S_k}\right)P_k^- \qquad\blacksquare$$

### Part 3 — Predictive measurement distribution

**Claim:** $p(y_k^- \mid Y_{k-1}) = \mathcal{N}(h\,m_k^-,\; h^2P_k^-+r)$.

**Proof:** This is exactly Part 2 of Q1 applied with $H=h$, $\Sigma_m=r$:
$$\mathbb{E}[y_k^-] = h\,m_k^-, \qquad \text{Var}(y_k^-) = h^2 P_k^- + r \qquad\blacksquare$$

Geometrically: before seeing the measurement, our best guess for what the sensor will read is $h$ times our state prediction, with inflated variance from both state uncertainty ($h^2P_k^-$) and sensor noise ($r$).

### Part 4 — Posterior-predictive measurement distribution

**Claim:** $p(y_k^- \mid Y_k) = \mathcal{N}(h\,m_k,\; h^2P_k+r)$.

**Proof:** After the update, $x_k^+ \sim \mathcal{N}(m_k, P_k)$. A new hypothetical draw $y_k^{\text{new}} = h x_k^+ + z$ (independent $z\sim\mathcal{N}(0,r)$) has distribution:
$$\mathbb{E}[y_k^\text{new}] = h\,m_k, \qquad \text{Var}(y_k^\text{new}) = h^2 P_k + r \qquad\blacksquare$$

This is narrower than the prior predictive because $P_k < P_k^-$.

### Part 5 — Numerical example and behaviour of the prior/posterior distributions

Take concrete parameter values $a=0.95,\ q=0.5,\ h=1,\ r=2$, with initial prior $x_0\sim\mathcal N(m_0=0,\,P_0=5)$.

**Step-by-step evolution (first 3 steps), using the recursions from Parts 1–2:**

**k = 1.** Suppose $y_1^{\mathrm{obs}} = 0.8$.

Prediction: $m_1^- = a\,m_0 = 0.95(0) = 0$, \ $P_1^- = a^2P_0 + q = 0.9025(5)+0.5 = 5.0125$.

Innovation variance: $S_1 = h^2P_1^- + r = 5.0125+2 = 7.0125$.

Gain: $K_1 = P_1^-h/S_1 = 5.0125/7.0125 \approx 0.7148$.

Update: $m_1 = 0 + 0.7148(0.8-0) \approx 0.5719$, \ $P_1 = (1-0.7148)(5.0125) \approx 1.4298$.

**k = 2.** Suppose $y_2^{\mathrm{obs}} = 1.1$.

Prediction: $m_2^- = 0.95(0.5719) \approx 0.5433$, \ $P_2^- = 0.9025(1.4298)+0.5 \approx 1.7909$.

$S_2 = 1.7909+2 = 3.7909$, \ $K_2 = 1.7909/3.7909 \approx 0.4724$.

$m_2 = 0.5433+0.4724(1.1-0.5433) \approx 0.8064$, \ $P_2 = (1-0.4724)(1.7909) \approx 0.9445$.

**k = 3.** Suppose $y_3^{\mathrm{obs}} = 0.95$.

Prediction: $m_3^- = 0.95(0.8064) \approx 0.7661$, \ $P_3^- = 0.9025(0.9445)+0.5 \approx 1.3527$.

$S_3 = 1.3527+2 = 3.3527$, \ $K_3 = 1.3527/3.3527 \approx 0.4035$.

$m_3 = 0.7661+0.4035(0.95-0.7661) \approx 0.8403$, \ $P_3 = (1-0.4035)(1.3527) \approx 0.8067$.

**Observed pattern (steady state):**

The variance sequence $P_0=5 \to P_1^-=5.0125 \to P_1=1.43 \to P_2^-=1.79 \to P_2=0.94 \to P_3^-=1.35 \to P_3=0.81 \to \dots$ is converging. Setting $P^-=P$ in steady state and solving the algebraic Riccati equation
$$P = a^2P - \frac{a^4 P^2}{a^2P+r} + q$$
for these parameter values gives a fixed point near $P_\infty \approx 0.74$, with steady-state gain $K_\infty \approx 0.39$.

**Qualitative description of the prior vs.\ posterior Gaussians at each step:**

- The **prior** (predictive) curve $\mathcal N(m_k^-, P_k^-)$ is always wider than the **posterior** curve $\mathcal N(m_k,P_k)$, because the update step strictly removes uncertainty: $P_k=(1-K_kh)P_k^- \le P_k^-$ whenever $0<K_kh\le 1$.
- The posterior mean $m_k$ always lies between the prior mean $m_k^-$ and the observation $y_k^{\mathrm{obs}}$, at a point determined by the gain $K_k$ — closer to whichever of the two is more certain.
- As $k$ increases, both $P_k^-$ and $P_k$ settle to constants ($P_\infty^-, P_\infty$), so the *shape* of the Gaussian (its spread) stops changing step to step, even though its *location* $m_k$ keeps tracking the incoming measurements.
- A sketch of this behaviour: at $k=0$ the prior is a wide bell curve centered at $0$; after each successive update the curve becomes narrower and re-centers closer to the cluster of measurements, until by around $k=10$–$15$ the width has stabilized.

**Summary — reading the prior/posterior picture described above:**

- Prior (dashed, wider) curve: belief *before* seeing $y_k^{\mathrm{obs}}$.
- Posterior (solid, narrower) curve: belief *after* assimilating $y_k^{\mathrm{obs}}$.
- The posterior is always at least as concentrated as the prior, and its mean is pulled toward the observed value by an amount controlled by the Kalman gain $K_k$.

---
# Q3 — 2-D Position Estimation

## Part A — Deriving the system matrices

### State and dynamics

State: $x_k = [p_x(k),\; p_y(k),\; v_x(k),\; v_y(k)]^\top$.

**Constant-velocity kinematics** over interval $\Delta t$:
$$p_x(k) = p_x(k-1) + \Delta t\,v_x(k-1), \quad v_x(k) = v_x(k-1)$$
and similarly in $y$. In matrix form $x_k = A x_{k-1}$:

$$A = \begin{bmatrix}1&0&\Delta t&0\\0&1&0&\Delta t\\0&0&1&0\\0&0&0&1\end{bmatrix}$$

**Derivation of A:** Each row encodes one kinematic equation. Row 1: $p_x(k)=1\cdot p_x(k-1)+0\cdot p_y(k-1)+\Delta t\cdot v_x(k-1)+0\cdot v_y(k-1)$. Rows 3–4: velocity unchanged → identity block. $\blacksquare$

---

### Measurement matrix

We observe only positions: $y_k = [p_x(k),\; p_y(k)]^\top$. So $y_k = H x_k$ with:

$$H = \begin{bmatrix}1&0&0&0\\0&1&0&0\end{bmatrix}$$

Row 1 picks out $p_x$, row 2 picks out $p_y$. Velocity components are unobserved. $\blacksquare$

---

### Noise input matrix

Model unmodeled acceleration as white noise $w_{k-1}=[a_x,\,a_y]^\top \sim \mathcal{N}(0,\Sigma_p)$. Over $\Delta t$:
$$p_x(k) = p_x(k-1)+\Delta t\,v_x(k-1) + \tfrac{1}{2}\Delta t^2 a_x, \quad v_x(k)=v_x(k-1)+\Delta t\,a_x$$

Reading off the coefficient of $[a_x,\,a_y]^\top$ in the state update:

$$G = \begin{bmatrix}\tfrac{1}{2}\Delta t^2 & 0 \\ 0 & \tfrac{1}{2}\Delta t^2 \\ \Delta t & 0 \\ 0 & \Delta t\end{bmatrix} \qquad\blacksquare$$

The induced state-space covariance is $Q = G\Sigma_p G^\top$.

## Part B — Filtering algorithm for noisy GPS measurements

**Task:** Develop a method to filter a sequence of noisy GPS position measurements $\{y_1^{\mathrm{obs}}, y_2^{\mathrm{obs}}, \dots\}$ using the 2-D constant-velocity model derived in Part A.

**Algorithm (written description):**

**Inputs:**
- Sequence of noisy GPS readings $y_k^{\mathrm{obs}} = [p_x^{\mathrm{meas}}(k),\,p_y^{\mathrm{meas}}(k)]^\top$ for $k=1,\dots,T$.
- Sampling interval $\Delta t$ (time between GPS fixes).
- Process-noise intensity $q$ (how much we expect unmodelled acceleration to perturb the trajectory) and measurement-noise variance $r^2$ (GPS receiver's quoted accuracy).
- Initial belief $m_0, P_0$ — if the initial position/velocity is unknown, use the first GPS reading for position and set velocity to zero with a *large* $P_0$ (e.g. $100\,I_4$) to reflect high initial uncertainty.

**Step 0 — Build the matrices** (as derived in Part A):
$$A=\begin{bmatrix}1&0&\Delta t&0\\0&1&0&\Delta t\\0&0&1&0\\0&0&0&1\end{bmatrix},\quad H=\begin{bmatrix}1&0&0&0\\0&1&0&0\end{bmatrix},\quad G=\begin{bmatrix}\tfrac12\Delta t^2&0\\0&\tfrac12\Delta t^2\\\Delta t&0\\0&\Delta t\end{bmatrix}$$
$$Q = G\,(qI_2)\,G^\top, \qquad R = r^2 I_2.$$

**Step 1 — Predict** (for each new time step $k$, before the GPS fix arrives):
$$m_k^- = A\,m_{k-1}, \qquad P_k^- = A P_{k-1} A^\top + Q.$$

**Step 2 — Update** (once $y_k^{\mathrm{obs}}$ is available):
$$S_k = H P_k^- H^\top + R,\qquad K_k = P_k^- H^\top S_k^{-1}$$
$$m_k = m_k^- + K_k\bigl(y_k^{\mathrm{obs}} - H m_k^-\bigr), \qquad P_k = (I-K_kH)\,P_k^-.$$

**Step 3 — Repeat** Steps 1–2 for $k=1,\dots,T$, storing $m_k$ (filtered position & velocity estimate) and $P_k$ (uncertainty) at each step.

**Output:** The sequence $\{m_k\}_{k=1}^T$ gives smoothed position estimates $(\hat p_x(k),\hat p_y(k))$ that are less noisy than the raw GPS fixes $y_k^{\mathrm{obs}}$, plus inferred velocity estimates $(\hat v_x(k),\hat v_y(k))$ that are not directly measured at all — they are recovered purely from the temporal correlation encoded in $A$.

**Why this works (intuition):** GPS noise is typically much larger than the smoothness implied by physical motion. By fusing the *physical model* (an object cannot teleport — its position next instant is constrained by its current velocity) with the *noisy sensor*, the filter produces an estimate that is smoother than the raw data and also estimates unmeasured quantities (velocity) for free.

**Practical notes:**
- If GPS dropouts occur (no reading at some $k$), simply skip the Update step for that $k$ and propagate only the Predict step — the covariance $P_k^-$ will grow, correctly reflecting increased uncertainty during the gap.
- $q$ and $r$ are tuning parameters: increasing $q$ relative to $r$ makes the filter trust the GPS more (track noisy data more closely); decreasing $q$ relative to $r$ makes the filter trust the constant-velocity model more (smoother but slower to react to genuine manoeuvres).